In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  

proxy = 'http://10.20.38.38:7890'
os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
import sys
sys.path.append('/home/ldy/Closed_loop_optimizing/model')
import torch
from PIL import Image
from torchvision import transforms
import torchvision.models as models
import torch.nn as nn
from einops.layers.torch import Rearrange
import math
import importlib
# import util
# importlib.reload(util)
import json
import open_clip
import torch.nn.functional as F
from torch.utils.data import DataLoader
import random
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [1]:
import os
import json
import re

def natural_sort_key(s):
    """自然排序键函数，用于正确处理数字顺序（如loop2, loop10）"""
    return [int(text) if text.isdigit() else text.lower() 
            for text in re.split('([0-9]+)', s)]

def collect_json_lists(root_dir):
    file_paths = []
    
    # 第一阶段：收集所有JSON文件路径
    for root, dirs, files in os.walk(root_dir):
        dirs.sort(key=natural_sort_key)  # 使用自然排序处理目录
        files.sort(key=natural_sort_key)  # 使用自然排序处理文件
        
        for file in files:
            if file.endswith('.json'):
                file_path = os.path.join(root, file)
                file_paths.append(file_path)
    
    # 第二阶段：按照要求的多级目录排序
    def sort_key(path):
        # 分解路径为各级目录
        parts = path.split(os.sep)
        # 获取上两级目录和上一级目录
        upper_dir = parts[-2] if len(parts) >= 2 else ''
        upper_upper_dir = parts[-3] if len(parts) >= 3 else ''
        return (natural_sort_key(upper_upper_dir), natural_sort_key(upper_dir))
    
    file_paths.sort(key=sort_key)
    
    # 第三阶段：按排序后的顺序处理文件
    combined_list = []
    for file_path in file_paths:
        print(f"file_path {file_path}")        
        try:                    
            with open(file_path, 'r') as f:
                data = json.load(f)
                if isinstance(data, list):
                    combined_list.extend(data)
        except (json.JSONDecodeError, IOError) as e:
            print(f"Error reading {file_path}: {e}")
    
    return combined_list

# 指定根目录
root_directory = '/mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/heuristic_generation/clip'

# 收集所有JSON文件中的列表内容
rating_rewards = collect_json_lists(root_directory)

# 打印结果或进行其他处理
print(f"总共收集了 {len(rating_rewards)} 个元素")

file_path /mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/heuristic_generation/clip/experiment_1/ratings.json
file_path /mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/heuristic_generation/clip/loop1/first_ten/ratings.json
file_path /mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/heuristic_generation/clip/loop2/fusion/ratings.json
file_path /mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/heuristic_generation/clip/loop2/greedy/ratings.json
file_path /mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/heuristic_generation/clip/loop3/fusion/ratings.json
file_path /mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/heuristic_generation/clip/loop3/greedy/ratings.json
file_path /mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/heuristic_generation/clip/loop4/fusion/ratings.json
file_path /mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/h

In [7]:
clip_rewards_path =  "/mnt/dataset0/xkp/closed-loop/server/xuruichen_psd_with_eeg_outputs/heuristic_generation/psd/all_chosen_rewards.npy"
clip_rewards = np.load(clip_rewards_path, allow_pickle=True)
clip_rewards.shape

(40,)

In [2]:
import numpy as np
from tqdm import tqdm
images_paths_cache = "/mnt/dataset0/xkp/closed-loop/server/xuruichen_clip_with_eeg_outputs/heuristic_generation/clip/viewed_image_paths.npy"
images_paths = np.load(images_paths_cache, allow_pickle=True)
len(images_paths)


64

In [8]:
max(clip_rewards)

0.8627214241093045

In [9]:
clip_rewards

array([0.7809643331248164, 0.7529999583027184, 0.6913802272124043,
       0.7024033746771292, 0.7206482075099999, 0.7142655264552206,
       0.6721276840264955, 0.6520309104356639, 0.7496747719216117,
       0.6898947754193031, 0.6584654140614054, 0.6486683663376462,
       0.821848612232547, 0.6828374180582887, 0.635982614609722,
       0.6324982521576042, 0.7421813234943566, 0.6949966746033571,
       0.6392561869017452, 0.6349234456566181, 0.7043503012766145,
       0.7029729021728784, 0.6456236506644861, 0.6453328193474739,
       0.8237640698060733, 0.7226425967391978, 0.6001394892996582,
       0.5348893342123049, 0.7750782311265463, 0.6482772698874645,
       0.6024216737649677, 0.5877233158996364, 0.7405393869639568,
       0.7066928487651905, 0.6568722855810081, 0.6471635391391425,
       0.8627214241093045, 0.8537876545889276, 0.7393010925780213,
       0.7167671156576882], dtype=object)

In [ ]:
# import os
# import numpy as np

# step_1_eegs_dir = "/mnt/dataset0/xkp/closed-loop/server/outputs/heuristic_generation/psd/loop1/first_ten"

# # 获取目录下所有.npy文件
# npy_files = [f for f in os.listdir(step_1_eegs_dir) if f.endswith('.npy')]

# # 按文件名排序（可选）
# npy_files.sort()

# # 初始化一个列表来存储加载的数组
# arrays = []

# # 遍历并加载所有.npy文件
# for file in npy_files:
#     file_path = os.path.join(step_1_eegs_dir, file)
#     arr = np.load(file_path)
#     arrays.append(arr)

# # 沿着第一个维度（batch维度）拼接所有数组

# step_1_eegs = np.asarray(arrays)

In [ ]:
# step_1_eegs.shape

(10, 64, 1250)

In [ ]:
bar = [
    [0.5400535643100739, 0.5326210737228394, 0.48303096294403075, 0.4724319875240326, 0.47124611735343935],
    [0.6865052580833435, 0.6614341338475546, 0.6511094172795614, 0.6766124765078226, 0.5994967818260193],
    [0.7649736523628230, 0.8102676033973690, 0.5639336228370660, 0.7781746029853810, 0.7761427164077750],
]

# 计算每组的 error（1 - similarity）
errors = [[1 - x for x in group] for group in bar]

# 基准组（第一组）的 errors
base_errors = errors[0]

# 计算相对 error（相对于基准组对应位置的 error）
relative_errors = []
for group in errors:
    relative_group = [(err / base_err) * 100 for err, base_err in zip(group, base_errors)]
    relative_errors.append(relative_group)

print("原始 error 值:")
for i, group in enumerate(errors):
    print(f"组 {i+1}: {group}")

print("\n相对 error（基准组=100%）:")
for i, group in enumerate(relative_errors):
    print(f"组 {i+1}: {group}")

In [ ]:
relative_errors

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_violin_results(x_labels, violin_data, colors=None, ylim=(0, 100), 
                       xlabel='Step', ylabel='Similarity', figsize=(6, 6), save_path=""):
    """
    Plot violin plot results showing distribution.
    
    Parameters:
    - x_labels: List of labels for the x-axis
    - violin_data: List of lists containing the results for each group
    - colors: List of colors for each violin (optional)
    - ylim: Tuple specifying y-axis limits (default: (0.4, 1.0))
    - xlabel: Label for x-axis (default: 'Step')
    - ylabel: Label for y-axis (default: 'Similarity')
    - figsize: Figure size (default: (8, 6))
    """
    # Calculate means for the dashed line
    means = [np.mean(group) for group in violin_data]
    
    # Set default colors if not provided
    if colors is None:
        colors = ['#e5a383', '#d45455', '#6a2a2b', '#b83f5b', '#f5768d']
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot violin plots
    parts = ax.violinplot(violin_data, showmeans=False, showmedians=False, showextrema=False)
    
    # Customize violin colors
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_edgecolor('black')
        pc.set_alpha(0.7)
    
    # Add individual data points with jitter
    for i, group in enumerate(violin_data):
        # Add jitter to x-position
        x_jitter = np.random.normal(i+1, 0.05, size=len(group))
        ax.scatter(x_jitter, group, color=colors[i], alpha=0.4, s=100, edgecolor='none')
    
    # Add horizontal dashed line at first group's y-value
    stim_baseline = means[0]
    ax.axhline(y=stim_baseline, color='black', linestyle='--', linewidth=1.5)
    
    # Set x-ticks and labels
    ax.set_xticks(np.arange(1, len(x_labels)+1))
    ax.set_xticklabels(x_labels, fontsize=22)
    
    # Modify y-ticks to show percentages (multiplied by 100)
    yticks = ax.get_yticks()
    ax.set_yticklabels([f'{int(y)}' for y in yticks], fontsize=20)
    
    # Set y-axis limits
    plt.ylim(*ylim)
    
    # Add label for the dashed line
    ax.text(0.02, stim_baseline + 0.01, 'Stim. baseline', 
            transform=ax.get_yaxis_transform(), 
            fontsize=20, color='black', va='bottom')
    
    # Remove right and top spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    # Add "N=5 Targets" label in upper right corner
    ax.text(0.88, 0.92, 'N=5 Targets', 
            transform=ax.transAxes,
            fontsize=20, color='black',
            ha='right', va='top')
    
    # Add labels
    plt.xlabel(xlabel, fontsize=29)
    plt.ylabel(ylabel, fontsize=26)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    # Adjust layout and show
    plt.tight_layout()
    plt.show()
    return fig, ax

# Example usage:
if __name__ == "__main__":
    # Example data (replace with your actual data)
    # relative_errors = [
    #     np.random.normal(30, 5, 20),  # Random
    #     np.random.normal(45, 8, 20),  # Offline
    #     np.random.normal(25, 6, 20)   # Closed-loop
    # ]
    
    colors = ['#e5a383', '#d45455', '#6a2a2b', '#b83f5b', '#f5768d']
    x_labels = ['Random', 'Offline', 'Closed-loop']
    
    fig, ax = plot_violin_results(x_labels, 
                                xlabel='', 
                                ylabel='Mean absolute error ratio (%)', 
                                violin_data=relative_errors, 
                                colors=colors, 
                                ylim=(0, 100), 
                                save_path="/home/ldy/Closed_loop_optimizing/Heuristic_generation/plots/violin_performance.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_boxplot_results(x_labels, box_data, colors=None, ylim=(0, 100), 
                         xlabel='Step', ylabel='Similarity', figsize=(6, 6), save_path=""):
    """
    Plot boxplot results with additional features.
    
    Parameters:
    - x_labels: List of labels for the x-axis
    - box_data: List of lists containing the results for each group
    - colors: List of colors for each box (optional)
    - ylim: Tuple specifying y-axis limits (default: (0, 100))
    - xlabel: Label for x-axis (default: 'Step')
    - ylabel: Label for y-axis (default: 'Similarity')
    - figsize: Figure size (default: (6, 6))
    """
    # Calculate means for the dashed line
    means = [np.mean(group) for group in box_data]
    
    # Set default colors if not provided
    if colors is None:
        colors = ['#e5a383', '#d45455', '#6a2a2b', '#b83f5b', '#f5768d']
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Create boxplot
    boxprops = dict(linewidth=2)
    whiskerprops = dict(linewidth=2)
    capprops = dict(linewidth=2)
    medianprops = dict(linewidth=2, color='black')
    
    bp = ax.boxplot(box_data, patch_artist=True, widths=0.6,
                    boxprops=boxprops, whiskerprops=whiskerprops,
                    capprops=capprops, medianprops=medianprops)
    
    # Set colors for boxes
    for i, box in enumerate(bp['boxes']):
        box.set(facecolor=colors[i], alpha=0.7)
    
    # Add horizontal dashed line at first box's mean y-value
    stim_baseline = means[0]
    ax.axhline(y=stim_baseline, color='black', linestyle='--', linewidth=1.5)
    
    # Set x-axis labels
    ax.set_xticklabels(x_labels)
    
    # Modify y-ticks to show percentages (multiplied by 100)
    yticks = ax.get_yticks()
    ax.set_yticklabels([f'{int(y)}' for y in yticks], fontsize=20)
    
    # Set y-axis limits (still in original scale, but labels show *100)
    plt.ylim(*ylim)
    
    # Add label for the dashed line (now using the original y-scale)
    ax.text(0.02, stim_baseline + 0.01, 'Stim. baseline', 
            transform=ax.get_yaxis_transform(), 
            fontsize=20, color='black', va='bottom')
    
    plt.xticks(fontsize=22)
    
    # Remove right and top spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    # Add "N=5 Targets" label in upper right corner
    ax.text(0.88, 0.92, 'N=5 Targets', 
            transform=ax.transAxes,
            fontsize=20, color='black',
            ha='right', va='top')
    
    # Add labels
    plt.xlabel(xlabel, fontsize=29)
    plt.ylabel(ylabel, fontsize=26)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    # Adjust layout and show
    plt.tight_layout()
    plt.show()
    return fig, ax

# Example usage:
if __name__ == "__main__":
    # Example data (replace with your actual data)
    # relative_errors = [
    #     np.random.normal(40, 10, 20),  # Random
    #     np.random.normal(30, 8, 20),   # Offline
    #     np.random.normal(20, 5, 20)    # Closed-loop
    # ]
    
    colors = ['#e5a383', '#d45455', '#6a2a2b', '#b83f5b', '#f5768d']
    x_labels = ['Random', 'Offline', 'Closed-loop']
    
    fig, ax = plot_boxplot_results(x_labels, 
                                  xlabel='', 
                                  ylabel='Mean absolute error ratio (%)', 
                                  box_data=relative_errors, 
                                  colors=colors, 
                                  ylim=(0, 100), 
                                  save_path="/home/ldy/Closed_loop_optimizing/Heuristic_generation/plots/boxplot_performance.png")